# Energia gazdálkodás optimalizálása Spark használatával

MySql adatbázisban gyűjtünk adatokat rendszerünk fogyasztásáról, a rendelkezésre álló napelemek termeléséről, illetve az energia ár alakulásáról 15 perces felbontásban.

Az árakat a hupx oldalról API-n keresztül szereztük.

A fogyasztás, termelés és ár adatok rendre a **consumer_profile**, **solar_panel_production** és **hupx_energy_price** táblákban kerülnek letárolásra időbélyeggel együt.



# Csatlakozás az adatbázisra

In [1]:
import findspark
import pyspark.sql
from pyspark.sql import SparkSession


findspark.init()

spark = SparkSession.builder \
    .appName("PySpark MySQL Connection") \
    .config("spark.jars", "/usr/share/java/mysql-connector-java-8.3.0.jar") \
    .config("spark.driver.extraClassPath", "/usr/share/java/mysql-connector-java-8.3.0.jar")\
    .getOrCreate()

mysql_host = "localhost"
mysql_port = "3306"
mysql_database = "energycom"
mysql_user = "root"
mysql_password = "mysql"

# MySQL JDBC URL
mysql_url = "jdbc:mysql://{0}:{1}/{2}?user={3}&password={4}" \
    .format(mysql_host, mysql_port, mysql_database, mysql_user, mysql_password)


24/02/29 14:23:12 WARN Utils: Your hostname, PySpark resolves to a loopback address: 127.0.1.1; using 192.168.64.128 instead (on interface ens33)
24/02/29 14:23:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
24/02/29 14:23:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


# Manuális ETL
Ez a blokk tesztadatok manuális betöltését szolgálja fáljokból, valamint egyéb kiegészítő adatok szolgáltatását, ha szükséges.

Itt kell elvégezni az adatok transzformálását, (pl: megfelelő típusra való cast-olás).

A következő blokkok futtatása kihagyható, ha nincsenek ilyen igények.

# Kiolvasás

In [ ]:

#reading and transforming csv input, only run this block when needed.
df = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("./dummy_data/consumption_dummy.csv")






# Transzformálás

In [ ]:
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "dd.MM.yyyy HH:mm"))
#df.printSchema()
#df.show()

# Betöltés

In [ ]:
#inserting data into DB
df.write\
  .format("jdbc")\
  .mode("append")\
  .option("driver","com.mysql.jdbc.Driver")\
  .option("url", "jdbc:mysql://localhost:3306/energycom")\
  .option("dbtable", "consumer_profile")\
  .option("user", "root")\
  .option("password", "mysql")\
  .save()

# Adatok beolvasása az adatbázisból

In [2]:
# Adatok betöltése a MySQL adatbázisból
consumer = spark.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", "consumer_profile") \
    .load()

production = spark.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", "solar_panel_production") \
    .load()

price = spark.read \
    .format("jdbc") \
    .option("url", mysql_url) \
    .option("driver", "com.mysql.cj.jdbc.Driver")\
    .option("dbtable", "hupx_energy_price") \
    .load()

consumer.show(10)
production.show(10)
price.show(10)

+----------+-------------------+---------------+
|profile_id|          timestamp|consumption_kwh|
+----------+-------------------+---------------+
|         1|2023-01-01 00:00:00|        0.02656|
|         1|2023-01-01 00:15:00|        0.02601|
|         1|2023-01-01 00:30:00|        0.02552|
|         1|2023-01-01 00:45:00|        0.02483|
|         1|2023-01-01 01:00:00|        0.02421|
|         1|2023-01-01 01:15:00|        0.02367|
|         1|2023-01-01 01:30:00|        0.02321|
|         1|2023-01-01 01:45:00|        0.02281|
|         1|2023-01-01 02:00:00|        0.02246|
|         1|2023-01-01 02:15:00|        0.02214|
+----------+-------------------+---------------+
only showing top 10 rows

+---------+-------------------+--------------+
|device_id|          timestamp|production_kwh|
+---------+-------------------+--------------+
|        1|2023-01-01 00:00:00|       0.00000|
|        1|2023-01-01 00:15:00|       0.00000|
|        1|2023-01-01 00:30:00|       0.00000|
|     

# Kiindulási adatok egyesítése

Az optimalizáció előkészítéséhez egyesítjük az adatokat egyetlen Pandas DataFrame-ben.

Előzetes szűrés szükséges a fogyasztó rendszer és a termelő egységre, az adatbázisban több egység termelési illetve fogyasztási adatait is tárolhatjuk. A szűrést azonosítók segítségével tehetjük meg.
Egységesség érdekében float típusú mezővé alakítom a három mutatót a számítási feladatok érdekében.

Itt kerülnek létrehozásra az általánosan szükséges paraméterek is, mint például a rendelkezésre álló akkumulátor kapacitása is.

In [4]:
from pyspark.sql.functions import *
# paramaméterek
capacity = 10 #kwh

# szűrés
consumer   = consumer.filter(col("profile_id") == 1)
production = production.filter(col("device_id") == 1)

# összekapcsolás
process = consumer.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#rendezés join után és cast-olás
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()
process.head()


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2023-01-01 00:00:00,0.02656,0.0,19.76
1,2023-01-01 00:15:00,0.02601,0.0,19.76
2,2023-01-01 00:30:00,0.02552,0.0,19.76
3,2023-01-01 00:45:00,0.02483,0.0,19.76
4,2023-01-01 01:00:00,0.02421,0.0,0.19


# Optimalizálási feladat

Most, hogy kész a rendezett adatsorunk, alkalmazhatjuk a különböző optimalizációs stratégiákat a minél eredményesebb energia gazdálkodás érdekében. 

A célfüggvény amit választottunk az a fogyasztási igények kiszolgálása a **költség minimalizálásával**. Annál jobb az eredmény, minél kevesebbe került az energiaszolgáltatás az adott időszakra.

különböző megvalósításokat igyekszünk összehasonlítani, hogy megállapíthassuk, melyik megoldás a legkedvezőbb.

# Egyszerű stratégia

A következő blokkokban egy próba algoritmust implementáltam, mely nem az adatok viszonya alapján készíti el a költség tervet.

Csupán referenciaként szolgáló megoldás.

**Működés:**

Ha az adott időben a termelés fedezi a fogyasztást, akkor többlet keletkezik, a többletből betáplálunk annyit az akkumulátorba, amennyi belefér, az el nem tárolható mennyiséget pedig kitápláljuk a hálózatra.

Másik esetben, mikor nem fedezi a termelés a fogyasztást, akkor a szükséges energiamennyiséget első sorban az akkumulátorban tárolt energiából pótoljuk, hogyha az sem elég, akkor vételezünk a hálózatról.


In [6]:
import numpy as np
# energia többlet és igény kiszámítása a termelés és fogyasztás különbségéből,
# valamint töltöttséget követő oszlopok bevezetése a negyedóra elejére és végére, alapértelmezett értékadás.
basic_p = process.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, x['production_kwh'] - x['consumption_kwh'], 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))
basic_p.head()

#from pyspark.sql.functions import when
#basic_p = process.withColumn("energy_to_store", when(col("production_kwh")-col("consumption_kwh")>0,col("production_kwh")-col("consumption_kwh"))\
#                          .otherwise(0))\
#               .withColumn("energy_to_get",when(col("consumption_kwh")-col("production_kwh")>0,col("consumption_kwh")-col("production_kwh"))\
#                          .otherwise(0))\
#               .withColumn("battery_charge_at_start",lit(0.00000))\
#               .withColumn("battery_charge_at_end",lit(0.00000))

# kezdő értéket állíthatunk az akkumulátorunknak a vizsgált időszak elején.
# basic_p.at[0,"battery_charge_start"] = 2

#ha tárolni kell akkor mennyit tudunk eltárolni
if basic_p.at[0,'energy_to_store'] > 0:
        temp = basic_p.at[0,'battery_charge_start'] + basic_p.at[0,'energy_to_store']
        if temp <= capacity:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = capacity
            
#ha be kell szerezni, mennyit tudunk szolgáltatni saját tárunkból
else:
        temp = basic_p.at[0,'battery_charge_start'] - basic_p.at[0,'energy_to_get']
        if temp > 0:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = 0

for i in range(1,int(basic_p.size/8)):
    basic_p.at[i,'battery_charge_start'] = basic_p.at[i-1,'battery_charge_end']
    if basic_p.at[i,'energy_to_store'] > 0:
        temp = basic_p.at[i,'battery_charge_start'] + basic_p.at[i,'energy_to_store']
        if temp <= capacity:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = capacity
    else:
        temp = basic_p.at[i,'battery_charge_start'] - basic_p.at[i,'energy_to_get']
        if temp > 0:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = 0
#basic_p.tail(10)

#az előző adatok alapján megállapítani, hogy mennyit fogyasztunk saját, illetve hálózati energiából, 
# valamint mennyi energiát táplálunk be saját akkumulátorunkba, vagy a hálózatba. 
basic_p = basic_p.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                       feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                       taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                       feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                       price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.001) # mwh-->kwh konverzió

basic_p.tail()


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,energy_to_store,energy_to_get,battery_charge_start,battery_charge_end,taking_from_battery,feeding_battery,taking_from_grid,feeding_grid,price
2971,2023-01-31 22:45:00,0.03004,0.0,121.650002,0.0,0.03004,8.86815,8.83811,0.03004,0.0,0.0,0.0,0.0
2972,2023-01-31 23:00:00,0.02862,0.0,98.690002,0.0,0.02862,8.83811,8.80949,0.02862,0.0,0.0,0.0,0.0
2973,2023-01-31 23:15:00,0.02734,0.0,98.690002,0.0,0.02734,8.80949,8.78215,0.02734,0.0,0.0,0.0,0.0
2974,2023-01-31 23:30:00,0.02614,0.0,98.690002,0.0,0.02614,8.78215,8.75601,0.02614,0.0,0.0,0.0,0.0
2975,2023-01-31 23:45:00,0.02507,0.0,98.690002,0.0,0.02507,8.75601,8.73094,0.02507,0.0,0.0,0.0,0.0
